# Kaggriculture — Advanced Farmers V1

**Google Colab notebook | Original Kaggle competition agent**

Crops-first baseline: fixed-priority tasks, Manhattan routing, wheat/carrot crop planning, and selling.

This notebook preserves the complete `V1` agent code from your supplied ZIP. You can edit the strategy, test it, and generate a Kaggle-ready `submission.tar.gz` without setting up a local Python project.

**How to use:** Open this `.ipynb` file in [Google Colab](https://colab.research.google.com/), then choose **Runtime → Run all**, or execute the cells in order. The Kaggle environment simulation cells are optional and can take some time. No competition dataset upload is required for the agent's basic operation.

> **Note:** The Python source is written to `/content/main.py` using several `%%writefile` cells. The first cell creates/overwrites the file, and subsequent cells append. Run these cells **in order** if you edit the agent. Your Kaggle submission uses `main.py` at the archive root.

## 1. Build the agent source file

### 1. Agent configuration, helpers and data models

In [ ]:
%%writefile /content/main.py
"""Advanced Farmers V1: a robust crops-first Kaggriculture agent.

The submission is intentionally self-contained and standard-library only.  The
planner separates WHAT should be done (TaskScheduler) from HOW a worker gets
there (ManhattanRouter).  It uses wheat and carrots, two inexpensive daily
hands, conservative seed inventory, daily crop care, and continuous selling.
"""

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple


CROPS: Dict[str, Dict[str, int]] = {
    "WHEAT": {"seed": 10, "first": 2, "harvest": 4, "yield": 4, "base": 25},
    "CARROT": {"seed": 20, "first": 2, "harvest": 3, "yield": 3, "base": 35},
}
PRODUCTS = ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER")
DEFAULT_SEASON_DAYS = 30
TARGET_PLOTS = 20
TARGET_HANDS = 2
SHED_SOFT_LIMIT = 70
SELL_FLOOR_RATIO = 0.60

Position = Tuple[int, int]
Action = List[Any]


def _get(obj: Any, key: str, default: Any = None) -> Any:
    """Read dict-like and attribute-like observations without raising."""
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def _as_dict(value: Any) -> Dict[str, Any]:
    return value if isinstance(value, dict) else {}


@dataclass(frozen=True)
class Worker:
    index: int
    position: Position
    inventory: Dict[str, int]


@dataclass(frozen=True)
class Task:
    priority: int
    position: Position
    action: Action
    name: str




### 2. Parse the game observation

In [ ]:
%%writefile -a /content/main.py
class StateView:
    """Defensive parser for the public/private Kaggriculture observation."""

    def __init__(self, obs: Any):
        self.raw = obs
        self.player = int(_get(obs, "player", 0) or 0)
        self.step = int(_get(obs, "step", 0) or 0)
        self.day = int(_get(obs, "day", self.step // 24) or 0)
        self.hour = int(_get(obs, "hour", self.step % 24) or 0)

        farms = _get(obs, "farms", []) or []
        self.valid = bool(farms) and 0 <= self.player < len(farms)
        self.farm = farms[self.player] if self.valid else {}
        self.tiles = _get(self.farm, "tiles", []) or []
        self.board_size = len(self.tiles) if self.tiles else 10
        self.money = float(_get(self.farm, "money", 0) or 0)

        private = _get(obs, "private", {}) or {}
        self.shed = _as_dict(_get(private, "shed", {}) or {})
        self.seeds = _as_dict(_get(private, "seeds", {}) or {})
        inventories = _get(private, "inventories", []) or []

        positions = [_get(self.farm, "farmer", [0, 0])]
        positions.extend(_get(self.farm, "hands", []) or [])
        self.workers: List[Worker] = []
        for idx, raw_pos in enumerate(positions):
            try:
                pos = (int(raw_pos[0]), int(raw_pos[1]))
            except (TypeError, ValueError, IndexError):
                pos = (0, 0)
            inv = _as_dict(inventories[idx]) if idx < len(inventories) else {}
            self.workers.append(Worker(idx, pos, inv))

        market = _get(obs, "market", {}) or {}
        self.prices = _as_dict(_get(market, "prices", {}) or {})
        town = _get(obs, "town", {}) or {}
        self.shops = list(_get(town, "unlocked_shops", []) or [])
        self.hires_today = int(_get(self.farm, "hires_today", 0) or 0)

    def iter_tiles(self) -> Iterable[Tuple[int, int, Any]]:
        for y, row in enumerate(self.tiles):
            for x, tile in enumerate(row):
                yield x, y, tile

    def tile_at(self, pos: Position) -> Any:
        x, y = pos
        if 0 <= y < len(self.tiles) and 0 <= x < len(self.tiles[y]):
            return self.tiles[y][x]
        return "LOCKED"

    def shed_access(self) -> List[Position]:
        half = self.board_size // 2
        return [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]

    def plant_counts(self) -> Dict[str, int]:
        counts = {crop: 0 for crop in CROPS}
        for _, _, tile in self.iter_tiles():
            if isinstance(tile, dict) and tile.get("kind") == "PLANT":
                crop = tile.get("crop")
                if crop in counts:
                    counts[crop] += 1
        return counts




### 3. Movement and crop planning

In [ ]:
%%writefile -a /content/main.py
class ManhattanRouter:
    """Deterministic one-step routing; locked tiles remain traversable."""

    MOVES = {
        (1, 0): ["EAST"],
        (-1, 0): ["WEST"],
        (0, 1): ["SOUTH"],
        (0, -1): ["NORTH"],
    }

    @staticmethod
    def distance(a: Position, b: Position) -> int:
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def step(self, source: Position, target: Position) -> Action:
        sx, sy = source
        tx, ty = target
        # Horizontal first is stable and works because units/structures do not block.
        if sx < tx:
            return self.MOVES[(1, 0)]
        if sx > tx:
            return self.MOVES[(-1, 0)]
        if sy < ty:
            return self.MOVES[(0, 1)]
        if sy > ty:
            return self.MOVES[(0, -1)]
        return ["PASS"]


class CropPlanner:
    """Creates a conservative 40% wheat / 60% carrot field plan."""

    @staticmethod
    def planned_crop(pos: Position) -> str:
        x, y = pos
        return "WHEAT" if (x + 2 * y) % 5 < 2 else "CARROT"

    @staticmethod
    def can_finish(crop: str, day: int) -> bool:
        return day + CROPS[crop]["harvest"] < DEFAULT_SEASON_DAYS

    def desired_seed_buys(self, state: StateView) -> Dict[str, int]:
        if state.day >= DEFAULT_SEASON_DAYS - 3:
            return {}
        counts = state.plant_counts()
        desired = {"WHEAT": 0, "CARROT": 0}
        for x, y, tile in state.iter_tiles():
            if tile is None or (isinstance(tile, dict) and tile.get("kind") == "WEED"):
                crop = self.planned_crop((x, y))
                if self.can_finish(crop, state.day):
                    desired[crop] += 1

        # Never expand beyond TARGET_PLOTS. Existing crops and held seeds both
        # count toward the target so repeated turns cannot overbuy.
        room = max(0, TARGET_PLOTS - sum(counts.values()) - sum(int(state.seeds.get(c, 0) or 0) for c in CROPS))
        buys: Dict[str, int] = {}
        for crop in ("CARROT", "WHEAT"):
            if room <= 0:
                break
            have = counts[crop] + int(state.seeds.get(crop, 0) or 0)
            quota = 12 if crop == "CARROT" else 8
            amount = min(room, desired[crop], max(0, quota - have))
            affordable = int(max(0, state.money - 200) // CROPS[crop]["seed"])
            amount = min(amount, affordable)
            if amount > 0:
                buys[crop] = amount
                room -= amount
        return buys




### 4. Market strategy

In [ ]:
%%writefile -a /content/main.py
class MarketStrategy:
    """Small, predictable market policy with guaranteed late liquidation."""

    def orders(self, state: StateView, seed_buys: Dict[str, int]) -> List[Action]:
        orders: List[Action] = []

        # Two cheap hands provide ample capacity for 20 plots. HIRE is atomic and
        # processed in list order, so both can be requested together at hour 0.
        if state.hour <= 1 and state.hires_today < TARGET_HANDS:
            for _ in range(TARGET_HANDS - state.hires_today):
                orders.append(["HIRE"])

        shed_total = sum(max(0, int(v or 0)) for v in state.shed.values())
        force_sell = state.day >= 26 or shed_total >= SHED_SOFT_LIMIT
        for item in PRODUCTS:
            amount = int(state.shed.get(item, 0) or 0)
            if amount <= 0:
                continue
            price = float(state.prices.get(item, 0) or 0)
            base = CROPS[item]["base"] if item in CROPS else price
            if force_sell or price >= base * SELL_FLOOR_RATIO:
                orders.append(["SELL", item, amount])

        # Seed orders come after sales so proceeds can fund them on the same turn.
        for crop in ("CARROT", "WHEAT"):
            amount = int(seed_buys.get(crop, 0) or 0)
            if amount > 0:
                orders.append(["BUY_SEED", crop, amount])

        return orders[:10]




### 5. Task assignment

In [ ]:
%%writefile -a /content/main.py
class TaskScheduler:
    """Build tasks, assign distinct targets, then route each worker one step."""

    def __init__(self, crop_planner: CropPlanner, router: ManhattanRouter):
        self.crop_planner = crop_planner
        self.router = router

    def _tasks(self, state: StateView) -> List[Task]:
        tasks: List[Task] = []
        plant_count = sum(state.plant_counts().values())

        for x, y, tile in state.iter_tiles():
            pos = (x, y)
            if isinstance(tile, dict) and tile.get("kind") == "PLANT":
                crop = tile.get("crop")
                if crop not in CROPS:
                    continue
                age = state.day - int(tile.get("planted_day", state.day))
                watered = bool(tile.get("watered_today", False))
                consecutive = int(tile.get("consecutive_unwatered", 0) or 0)
                units = int(tile.get("yield_units", 0) or 0)

                # A plant with one missed day is at immediate risk at day-end.
                if not watered and consecutive >= 1:
                    tasks.append(Task(0, pos, ["WATER"], "rescue-water"))
                # Endgame takes any mature output; otherwise wait for peak yield.
                harvest_age = CROPS[crop]["first"] if state.day >= 28 else CROPS[crop]["harvest"]
                if units > 0 and age >= harvest_age:
                    tasks.append(Task(1, pos, ["HARVEST"], "harvest"))
                if not watered:
                    tasks.append(Task(2, pos, ["WATER"], "water"))

            elif isinstance(tile, dict) and tile.get("kind") == "WEED":
                tasks.append(Task(4, pos, ["DIG"], "dig"))

            elif tile is None and plant_count < TARGET_PLOTS and state.hour <= 14:
                crop = self.crop_planner.planned_crop(pos)
                if int(state.seeds.get(crop, 0) or 0) > 0 and self.crop_planner.can_finish(crop, state.day):
                    tasks.append(Task(3, pos, ["PLANT", crop], "plant"))

        # Carried output is deposited late in the day or in useful-sized batches.
        for worker in state.workers:
            carried = sum(max(0, int(v or 0)) for v in worker.inventory.values())
            if carried and (carried >= 6 or state.hour >= 18 or state.day >= 28):
                nearest = min(state.shed_access(), key=lambda p: self.router.distance(worker.position, p))
                tasks.append(Task(3, nearest, ["DROP"], f"drop-{worker.index}"))

        return tasks

    def actions(self, state: StateView) -> List[Action]:
        if not state.workers:
            return []
        tasks = self._tasks(state)
        actions: List[Action] = [["PASS"] for _ in state.workers]
        reserved_positions: set = set()
        reserved_seed = {crop: 0 for crop in CROPS}

        # Urgency dominates, then distance, then stable coordinates/name.
        for worker in state.workers:
            candidates: List[Task] = []
            for task in tasks:
                if task.position in reserved_positions:
                    continue
                if task.name.startswith("drop-") and task.name != f"drop-{worker.index}":
                    continue
                if task.action[0] == "PLANT":
                    crop = task.action[1]
                    available = int(state.seeds.get(crop, 0) or 0) - reserved_seed[crop]
                    if available <= 0:
                        continue
                candidates.append(task)
            if not candidates:
                continue
            chosen = min(
                candidates,
                key=lambda t: (
                    t.priority,
                    self.router.distance(worker.position, t.position),
                    t.position[1],
                    t.position[0],
                    t.name,
                ),
            )
            reserved_positions.add(chosen.position)
            if chosen.action[0] == "PLANT":
                reserved_seed[chosen.action[1]] += 1
            actions[worker.index] = chosen.action if worker.position == chosen.position else self.router.step(worker.position, chosen.position)

        return actions




### 6. Kaggle agent entry point

In [ ]:
%%writefile -a /content/main.py
ROUTER = ManhattanRouter()
CROP_PLANNER = CropPlanner()
MARKET = MarketStrategy()
SCHEDULER = TaskScheduler(CROP_PLANNER, ROUTER)


def agent(obs: Any) -> Dict[str, Any]:
    """Kaggle entry point. Always returns a legal, complete action structure."""
    try:
        state = StateView(obs)
        if not state.valid:
            return {"farmer": ["PASS"], "hands": [], "market": []}
        unit_actions = SCHEDULER.actions(state)
        if not unit_actions:
            unit_actions = [["PASS"]]
        seed_buys = CROP_PLANNER.desired_seed_buys(state)
        market_orders = MARKET.orders(state, seed_buys)
        return {
            "farmer": unit_actions[0],
            "hands": unit_actions[1:],
            "market": market_orders,
        }
    except Exception:
        # Competition validation penalizes crashes much more than a single PASS.
        return {"farmer": ["PASS"], "hands": [], "market": []}



## 2. Validate the Python agent

This cell checks that `main.py` exists, compiles, exposes the `agent(obs)` entry point, and responds safely to an empty observation. It does **not** require Kaggle packages.

In [ ]:
from pathlib import Path
import ast
import importlib.util
import sys

agent_file = Path('/content/main.py')
source_text = agent_file.read_text(encoding='utf-8')
ast.parse(source_text)
compile(source_text, str(agent_file), 'exec')

spec = importlib.util.spec_from_file_location('kaggriculture_agent_local', agent_file)
agent_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = agent_module
spec.loader.exec_module(agent_module)
assert callable(agent_module.agent)
print('Agent file and entry point: OK')
print('Safe fallback action:', agent_module.agent({}))

## 3. Optional: run a Kaggriculture match

The source project's benchmark notes use `kaggle-environments==1.32.7`. Install that version here for a comparable environment. Installation and game runs require an internet-enabled Colab runtime and are **not needed to export the submission**.

In [ ]:
%pip -q install "kaggle-environments==1.32.7"

**Quick smoke test** against the competition starter agent (48 turns). This verifies that the agent can be loaded and play through a short game. Use the longer run below for a full season.

In [ ]:
from kaggle_environments import make

short_env = make('kaggriculture', configuration={'episodeSteps': 48}, debug=True)
short_env.run(['/content/main.py', 'starter'])
print('Last-turn results (status, reward):')
for player_index, result in enumerate(short_env.steps[-1]):
    print(f'  Player {player_index}:', result.status, result.reward)


**Full season (optional):** 720 turns, matching the source README benchmark setup.

In [ ]:
full_env = make('kaggriculture', configuration={'episodeSteps': 720}, debug=True)
full_env.run(['/content/main.py', 'starter'])
print('Full-season results (status, reward):')
for player_index, result in enumerate(full_env.steps[-1]):
    print(f'  Player {player_index}:', result.status, result.reward)


## 4. Export a Kaggle-ready submission

This creates a `.tar.gz` containing **exactly one file**, named `main.py` at the archive root (the structure specified in your original project's README), then downloads the archive in Colab. Run after the agent source cells; no game simulation is required.

The `.ipynb` file is for development in Colab. The downloaded `.tar.gz` file is for Kaggle competition submission.

In [ ]:
from pathlib import Path
import tarfile

main_path = Path('/content/main.py')
assert main_path.is_file(), 'Run the agent source cells first.'
archive_path = Path('/content/kaggriculture_v1_submission.tar.gz')
with tarfile.open(archive_path, mode='w:gz') as archive:
    archive.add(main_path, arcname='main.py')

with tarfile.open(archive_path, mode='r:gz') as archive:
    assert archive.getnames() == ['main.py']
print('Submission archive ready:', archive_path)

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print('Outside Colab: retrieve the file from', archive_path)
